In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

In [ ]:
import torch
import numpy as np
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio, DatasetDict
import matplotlib.pyplot as plt

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_or = AutoModelForAudioClassification.from_pretrained("openai/whisper-base").to(device).eval()
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/whisper-finetuned-TESS").to(device).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/whisper-finetuned-TESS")

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

In [ ]:
def extract_all_layers(model, ds):
    per_layer_embeddings = None
    with torch.no_grad():
        for sample in ds:
            audio = sample["audio"]
            inputs = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt")
            input_features = inputs["input_features"].to(device)

            outputs = model(input_features=input_features, output_hidden_states=True)
            hidden_states = outputs.hidden_states

            if per_layer_embeddings is None:
                per_layer_embeddings = [[] for _ in hidden_states]

            for layer_idx, h in enumerate(hidden_states):
                per_layer_embeddings[layer_idx].append(h.mean(dim=1).squeeze().cpu().numpy())

    return [np.array(layer) for layer in per_layer_embeddings]

In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    hsic = np.linalg.norm(Y.T @ X, ord="fro") ** 2
    norm_x = np.linalg.norm(X.T @ X, ord="fro")
    norm_y = np.linalg.norm(Y.T @ Y, ord="fro")
    return hsic / (norm_x * norm_y)

In [ ]:
print("Εξαγωγή embeddings (Pretrained)...")
layers_or = extract_all_layers(model_or, test_dataset)
print("Εξαγωγή embeddings (Fine-tuned)...")
layers_ft = extract_all_layers(model_ft, test_dataset)

In [ ]:
cka_scores = [linear_cka(layers_or[i], layers_ft[i]) for i in range(len(layers_or))]
for i, score in enumerate(cka_scores):
    print(f"Layer {i}: CKA = {score:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(range(len(cka_scores)), cka_scores, marker="o")
plt.xlabel("Layer"); plt.ylabel("Linear CKA")
plt.title("Layer-wise CKA: Pretrained vs Fine-tuned Whisper (TESS)")
plt.ylim(0, 1); plt.grid(alpha=0.3)
plt.show()